<a href="https://colab.research.google.com/github/aMDy0k/workspace/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import boon
import pandas as pd
import numpy as np
from boon import Demo
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform

In [ ]:
# 1. Загружаем демку и читаем таблицу урона за 1 проход
demo = Demo("97104015.dem")
demo.load("damage", "player_ticks")
df_damage = demo.damage.to_pandas()
df_pos = demo.player_ticks.to_pandas()

In [ ]:
print("Колонки в df_damage:", df_damage.columns.tolist())
print("\nКолонки в player_ticks:", df_pos.columns.tolist())
print("\nУникальные victim_class:", df_damage["victim_class"].dropna().unique()[:20])
print("\nПримеры victim_hero_id:", df_damage["victim_hero_id"].unique()[:10])
print("\nМин/макс тики:", df_damage["tick"].min(), df_damage["tick"].max())

Колонки в df_damage: ['tick', 'damage', 'pre_damage', 'victim_hero_id', 'attacker_hero_id', 'victim_health_new', 'hitgroup_id', 'crit_damage', 'attacker_class', 'victim_class']

Колонки в player_ticks: ['tick', 'hero_id', 'x', 'y', 'z', 'pitch', 'yaw', 'roll', 'in_regen_zone', 'in_item_shop', 'death_time', 'last_spawn_time', 'respawn_time', 'health', 'max_health', 'lifestate', 'souls', 'spent_souls', 'in_combat_end_time', 'in_combat_last_damage_time', 'in_combat_start_time', 'player_damage_dealt_end_time', 'player_damage_dealt_last_damage_time', 'player_damage_dealt_start_time', 'player_damage_taken_end_time', 'player_damage_taken_last_damage_time', 'player_damage_taken_start_time', 'time_revealed_by_npc', 'build_id', 'is_alive', 'has_rebirth', 'has_rejuvenator', 'has_ultimate_trained', 'health_regen', 'ultimate_cooldown_start', 'ultimate_cooldown_end', 'ap_net_worth', 'gold_net_worth', 'denies', 'hero_damage', 'hero_healing', 'objective_damage', 'self_healing', 'kill_streak', 'last_hi

In [ ]:
# 1. Берем тики лайнинга (2-я минута)
first_tick = df_pos["tick"].min()
df_pos["minute"] = ((df_pos["tick"] - first_tick) / (64 * 60)).astype(int)

m2_df = df_pos[
    (df_pos["minute"] == 2) &
    (df_pos["hero_id"] > 0) &
    (df_pos["is_alive"] == True)
]

# 2. Усредняем позиции по каждому hero_id
coords = m2_df.groupby("hero_id")[["x", "y"]].mean().reset_index()

print(f"Всего уникальных героев на карте: {len(coords)}")
print("\nСырые координаты героев (X, Y):")
print(coords)

Всего уникальных героев на карте: 12

Сырые координаты героев (X, Y):
    hero_id            x            y
0         1  -316.731262     1.547917
1         2  8028.802246   992.939819
2         7 -7635.986816 -1036.007935
3        10  7366.143066  -265.273468
4        15 -7419.442383   243.400848
5        16   -33.954597   225.920059
6        17   432.912415 -1219.484863
7        19  7581.324707  -196.558136
8        27 -7354.337402   161.208817
9        52 -7586.564941 -1034.565674
10       67  7691.166504  1037.831909
11       69  -425.470734 -1290.443237


In [ ]:
# 1. Отсчитываем минуты от первого урона
first_tick = df_damage["tick"].min()
df_damage["minute"] = ((df_damage["tick"] - first_tick) / (64 * 60)).astype(int)

# 2. Фильтруем: первые 10 минут, нападающий — герой (>0), цель — лайновый крип (class 4)
creep_df = df_damage[
    (df_damage["minute"] < 6) &
    (df_damage["victim_class"] == 4) &
    (df_damage["attacker_hero_id"] > 0)
].copy()

# 3. Временной ряд урона по минутам
time_series = creep_df.groupby(["attacker_hero_id", "minute"])["damage"].sum().unstack(fill_value=0)
total_damage = creep_df.groupby("attacker_hero_id")["damage"].sum().sort_values(ascending=False)

print("--- СУММАРНЫЙ УРОН ПО ЛАЙНОВЫМ КРИПАМ (0-6 МИН) ---")
print(total_damage)

print("\n--- ВРЕМЕННОЙ РЯД ПО МИНУТАМ ---")
print(time_series)

--- СУММАРНЫЙ УРОН ПО ЛАЙНОВЫМ КРИПАМ (0-10 МИН) ---
attacker_hero_id
10    6835
67    6112
17    5351
2     5244
15    4958
19    4914
16    4848
7     4814
6     4162
69    3999
1     3888
3     3468
Name: damage, dtype: int32

--- ВРЕМЕННОЙ РЯД ПО МИНУТАМ ---
minute               0     1     2     3     4     5
attacker_hero_id                                    
1                  277   217   525   189  1176  1504
2                  537   759   787  1008   859  1294
3                  589   649   290   947   575   418
6                  555   967  1662   311   517   150
7                  775   279   206  1884   952   718
10                 839  1260   785  1406   844  1701
15                 842   708   967   672   692  1077
16                1033  1009   698   973   475   660
17                 975   768   943   773  1268   624
19                 656  1034   685   579   882  1078
67                 968  1218   679   833  1517   897
69                1001   801   244   916   354  